# 🔄 Recursive Language Models (RLMs) — Implémentation éducative

> **Papier:** *Recursive Language Models* — Zhang, Kraska, Khattab (MIT CSAIL, Dec 2025)  
> **arXiv:** `2512.24601v1`

Ce notebook implémente pas-à-pas les concepts clés du papier pour comprendre **concrètement** comment fonctionne un RLM.

---

## Pourquoi ce papier est important

Les LLMs ont deux problèmes fondamentaux avec les longs contextes :
1. **Fenêtre de contexte limitée** — GPT-5 ne peut traiter que ~272K tokens
2. **Context rot** — même en-deçà de cette limite, la qualité se dégrade avec la longueur

**L'insight clé du RLM :** ne pas injecter le prompt dans le réseau de neurones, mais le traiter comme un **objet dans un environnement externe** que le LLM manipule via du code.

```
LLM classique:    Prompt ──────────► [Transformer] ──► Réponse
                  (limité par la fenêtre)

RLM:              Prompt ──► [Variable REPL] ◄──► [LLM écrit du code] ──► Réponse
                  (taille arbitraire)              │
                                                   ▼
                                              [Sous-LLM récursif]
```


---
## 1. Le LLM simulé

Dans une implémentation réelle, on ferait des appels API (OpenAI, Anthropic…).  
Ici, on simule un LLM avec une fenêtre de contexte limitée pour montrer les mécanismes.

L'important est **l'interface** : `string → string`, identique à un vrai LLM.


In [10]:
import json
import textwrap
import re
import io
import sys
import traceback
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
from collections import Counter

class SimulatedLLM:
    """
    Simule un LLM avec fenetre de contexte limitee.
    
    En realite, remplacez __call__ par:
        response = openai.chat.completions.create(model="gpt-5", messages=[...])
        return response.choices[0].message.content
    """
    
    def __init__(self, name: str = "simulated-llm", context_window: int = 4000):
        self.name = name
        self.context_window = context_window
        self.call_count = 0
        self.total_chars_processed = 0
    
    def __call__(self, prompt: str) -> str:
        self.call_count += 1
        self.total_chars_processed += len(prompt)
        
        if len(prompt) > self.context_window:
            return "[ERREUR: Prompt depasse la fenetre de contexte]"
        
        prompt_lower = prompt.lower()
        
        if "mot magique" in prompt_lower or "magic word" in prompt_lower:
            if "zephyr" in prompt_lower:
                return "Le mot magique trouve est: ZEPHYR"
            elif "aurora" in prompt_lower:
                return "Le mot magique trouve est: AURORA"
            return "Aucun mot magique trouve dans ce fragment."
        
        if "classifie" in prompt_lower or "categorie" in prompt_lower:
            categories = {}
            for line in prompt.split("\n"):
                ll = line.lower()
                if "combien" in ll or "quel nombre" in ll:
                    categories[line.strip()[:50]] = "numeric_value"
                elif "qui est" in ll or "quelle personne" in ll:
                    categories[line.strip()[:50]] = "human_being"
                elif "ou" in ll and ("se trouve" in ll or "est situe" in ll or "pays" in ll or "muraille" in ll):
                    categories[line.strip()[:50]] = "location"
                elif "qu'est-ce" in ll or "defini" in ll:
                    categories[line.strip()[:50]] = "description"
            if categories:
                return json.dumps(categories, ensure_ascii=False, indent=2)
            return "Aucune entree a classifier."
        
        if "resume" in prompt_lower:
            return f"Resume ({len(prompt)} chars): contenu traite."
        
        if "agrege" in prompt_lower or "synthetise" in prompt_lower:
            return "Synthese complete des fragments analyses."
        
        return f"[LLM a traite {len(prompt)} caracteres]"
    
    def stats(self) -> dict:
        return {
            "appels": self.call_count,
            "chars_traites": self.total_chars_processed,
            "fenetre": self.context_window,
        }

# Test rapide
llm = SimulatedLLM("test-llm", context_window=500)
print("Prompt court :", llm("Trouve le mot magique: ZEPHYR est cache ici"))
print("Prompt trop long:", llm("x" * 600))
print("Stats:", llm.stats())


Prompt court : Le mot magique trouve est: ZEPHYR
Prompt trop long: [ERREUR: Prompt depasse la fenetre de contexte]
Stats: {'appels': 2, 'chars_traites': 643, 'fenetre': 500}


---
## 2. L'environnement REPL — le cœur du RLM

C'est **l'innovation centrale** du papier. L'environnement REPL fournit :

| Composant | Rôle |
|-----------|------|
| `context` (variable) | Le prompt complet, stocké comme objet manipulable |
| `llm_query(prompt)` | Fonction pour faire des **sous-appels LLM récursifs** |
| `print()` | Renvoie les résultats au LLM racine |
| Namespace persistant | Les variables survivent entre les exécutions |

### Analogie out-of-core (du papier)

| Base de données | RLM |
|---|---|
| RAM | Fenêtre de contexte du LLM |
| Disque dur | Prompt complet (variable REPL) |
| Requête SQL | Code Python du LLM |
| Index B-tree | Regex / recherche par mots-clés |
| Jointure | Sous-appels LLM récursifs |


In [11]:
@dataclass
class REPLExecution:
    """Resultat d'une execution de code dans le REPL."""
    code: str
    stdout: str
    stderr: str
    success: bool


class REPLEnvironment:
    """
    Environnement REPL persistant pour le RLM.
    
    Le prompt est stocke dans `context`, le LLM ecrit du code
    qui s'execute dans cet environnement et peut appeler `llm_query()`.
    """
    
    def __init__(self, context: str, sub_llm: SimulatedLLM, max_output_chars: int = 2000):
        self.max_output_chars = max_output_chars
        self.execution_history: list[REPLExecution] = []
        self.sub_llm = sub_llm
        self.sub_call_count = 0
        
        # Namespace persistant
        self._namespace = {
            "context": context,           # Le prompt complet
            "llm_query": self._llm_query, # Sous-appels recursifs
            "re": re,
            "json": json,
            "Counter": Counter,
        }
    
    def _llm_query(self, prompt: str) -> str:
        """
        Fonction exposee au code du LLM pour les sous-appels.
        
        C'est ici que la RECURSION se produit:
        Le LLM racine ecrit du code -> le code appelle llm_query()
        -> un AUTRE LLM traite un fragment du contexte.
        """
        self.sub_call_count += 1
        return self.sub_llm(prompt)
    
    def execute(self, code: str) -> REPLExecution:
        """Execute du code Python, capture stdout."""
        stdout_buf = io.StringIO()
        stderr_buf = io.StringIO()
        old_out, old_err = sys.stdout, sys.stderr
        sys.stdout, sys.stderr = stdout_buf, stderr_buf
        
        success = True
        try:
            exec(code, self._namespace)
        except Exception:
            success = False
            traceback.print_exc(file=stderr_buf)
        finally:
            sys.stdout, sys.stderr = old_out, old_err
        
        stdout = stdout_buf.getvalue()
        stderr = stderr_buf.getvalue()
        
        if len(stdout) > self.max_output_chars:
            stdout = stdout[:self.max_output_chars] + f"\n... [tronque, {len(stdout)} chars total]"
        
        result = REPLExecution(code=code, stdout=stdout, stderr=stderr, success=success)
        self.execution_history.append(result)
        return result
    
    def get_variable(self, name: str):
        return self._namespace.get(name)

print("REPLEnvironment defini")


REPLEnvironment defini


### 🧪 Test de l'environnement REPL

Vérifions que le REPL fonctionne : on charge un texte comme `context`, on écrit du code pour l'explorer, et on fait un sous-appel LLM.


In [12]:
# Creer un environnement REPL avec un texte de test
test_context = """Chapitre 1: Le debut de l'aventure.
Il faisait nuit quand Alice trouva la cle magique.
Le mot magique secret est: ZEPHYR.
Chapitre 2: La suite des evenements.
Bob continua son chemin dans la foret."""

sub_llm = SimulatedLLM("sub-llm", context_window=2000)
repl = REPLEnvironment(context=test_context, sub_llm=sub_llm)

# Execution 1: Explorer la structure
result1 = repl.execute('''
print(f"Taille du contexte: {len(context)} chars")
print(f"Lignes: {len(context.splitlines())}")
print(f"Extrait: {context[:80]}...")
''')
print("=== Execution 1: Probing ===")
print(result1.stdout)

# Execution 2: Recherche regex + sous-appel LLM
result2 = repl.execute('''
import re
match = re.search(r"mot magique.*", context)
if match:
    fragment = match.group()
    # SOUS-APPEL RECURSIF au LLM
    answer = llm_query(f"Trouve le mot magique dans: {fragment}")
    print(f"Sous-LLM dit: {answer}")
''')
print("=== Execution 2: Regex + sous-appel recursif ===")
print(result2.stdout)

# Les variables persistent !
print(f"\nSous-appels LLM effectues: {repl.sub_call_count}")
print(f"Variable 'answer' dans REPL: {repl.get_variable('answer')}")


=== Execution 1: Probing ===
Taille du contexte: 197 chars
Lignes: 5
Extrait: Chapitre 1: Le debut de l'aventure.
Il faisait nuit quand Alice trouva la cle ma...

=== Execution 2: Regex + sous-appel recursif ===
Sous-LLM dit: Le mot magique trouve est: ZEPHYR


Sous-appels LLM effectues: 1
Variable 'answer' dans REPL: Le mot magique trouve est: ZEPHYR


---
## 3. Le Recursive Language Model complet

Le RLM orchestre la boucle itérative :

```
Pour chaque itération:
    1. Construire un prompt pour le LLM racine (métadonnées, PAS le contexte complet)
    2. Le LLM racine décide: écrire du code REPL ou donner une réponse finale
    3. Si code → exécuter dans le REPL → montrer la sortie au LLM racine → boucler
    4. Si FINAL() ou FINAL_VAR() → terminer et renvoyer la réponse
```

### Les patterns émergents (Section 3.1 du papier)

Le LLM développe **spontanément** ces stratégies sans entraînement spécifique :

1. **Probing** — `print(context[:200])` pour comprendre la structure
2. **Filtrage par priors** — regex avec des mots-clés basés sur ses connaissances
3. **Chunking + sous-appels** — découper et envoyer chaque morceau à un sous-LLM
4. **Agrégation programmatique** — combiner les résultats par code Python
5. **Vérification** — relancer un sous-appel pour confirmer la réponse


In [13]:
class RLMStatus(Enum):
    RUNNING = "running"
    FINAL_ANSWER = "final_answer"
    FINAL_VAR = "final_var"


@dataclass
class RLMStep:
    """Un pas dans la trajectoire du RLM."""
    iteration: int
    thought: str
    code: Optional[str]
    execution: Optional[REPLExecution]
    status: RLMStatus


@dataclass
class RLMResult:
    """Resultat final d'un appel RLM."""
    answer: str
    trajectory: list[RLMStep]
    stats: dict


class RecursiveLanguageModel:
    """
    Recursive Language Model (RLM).
    
    Interface externe IDENTIQUE a un LLM classique:
        rlm(prompt, query) -> reponse (string)
    
    Mais en interne, le prompt n'est jamais injecte directement.
    Il est manipule comme un objet dans un environnement REPL.
    """
    
    def __init__(self, root_llm, sub_llm, max_iterations=10, verbose=True):
        self.root_llm = root_llm
        self.sub_llm = sub_llm
        self.max_iterations = max_iterations
        self.verbose = verbose
    
    def __call__(self, prompt: str, query: str) -> RLMResult:
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"  RLM | {len(prompt):,} chars | Query: {query[:60]}...")
            print(f"  Root: {self.root_llm.name} (fenetre {self.root_llm.context_window:,})")
            print(f"  Sub:  {self.sub_llm.name}")
            print(f"{'='*60}")
        
        repl = REPLEnvironment(context=prompt, sub_llm=self.sub_llm)
        trajectory = []
        
        for i in range(self.max_iterations):
            thought, code_str, status, answer = self._decide(query, prompt, i, repl, trajectory)
            
            execution = None
            if code_str and status == RLMStatus.RUNNING:
                execution = repl.execute(code_str)
                if self.verbose:
                    self._show_step(i, thought, code_str, execution)
            
            step = RLMStep(i, thought, code_str, execution, status)
            trajectory.append(step)
            
            if status == RLMStatus.FINAL_ANSWER:
                if self.verbose:
                    print(f"\n  FINAL(): {answer}")
                return RLMResult(answer, trajectory, self._stats(repl))
            
            if status == RLMStatus.FINAL_VAR:
                val = repl.get_variable(answer)
                final = str(val) if val is not None else "[Variable introuvable]"
                if self.verbose:
                    print(f"\n  FINAL_VAR({answer}): {final[:200]}")
                return RLMResult(final, trajectory, self._stats(repl))
        
        return RLMResult("[Max iterations]", trajectory, self._stats(repl))
    
    def _decide(self, query, context, iteration, repl, prev):
        ql = query.lower()
        if "mot magique" in ql:
            return self._strat_niah(context, iteration, repl)
        elif "classifie" in ql or "categorie" in ql or "combien" in ql:
            return self._strat_aggregation(query, context, iteration)
        else:
            return self._strat_generic(query, context, iteration)
    
    # ── Strategie NIAH ──
    def _strat_niah(self, context, it, repl):
        if it == 0:
            return (
                "Probing: examiner la structure du contexte",
                'print(f"Taille: {len(context)} chars, {len(context.splitlines())} lignes")\n'
                'print(f"Debut: {context[:150]}...")\n'
                'print(f"Fin: ...{context[-150:]}")',
                RLMStatus.RUNNING, None
            )
        elif it == 1:
            return (
                "Filtrage par priors: recherche regex ciblee (pattern Figure 4a)",
                '''keywords = ["magique", "magic", "secret", "cache"]
for kw in keywords:
    idx = context.lower().find(kw)
    if idx != -1:
        snippet = context[max(0,idx-80):min(len(context),idx+80)]
        print(f"'{kw}' trouve a pos {idx}: ...{snippet}...")''',
                RLMStatus.RUNNING, None
            )
        elif it == 2:
            return (
                "Sous-appel recursif sur le fragment trouve (coeur du RLM)",
                '''import re
match = re.search(r"(?i)(mot magique|magic word)[^\\n]*", context)
if match:
    s = max(0, match.start() - 300)
    e = min(len(context), match.end() + 300)
    fragment = context[s:e]
    # SOUS-APPEL RECURSIF
    found_answer = llm_query(f"Trouve le mot magique dans:\n{fragment}")
    print(f"Sous-LLM -> {found_answer}")
else:
    chunk_size = len(context) // 5
    found_answer = "Non trouve"
    for i in range(5):
        chunk = context[i*chunk_size:(i+1)*chunk_size]
        r = llm_query(f"Cherche le mot magique: {chunk}")
        print(f"Chunk {i}: {r}")
        if "trouve" in r.lower():
            found_answer = r
            break''',
                RLMStatus.RUNNING, None
            )
        else:
            found = repl.get_variable("found_answer")
            if found:
                return ("Reponse trouvee via sous-appel", None, RLMStatus.FINAL_VAR, "found_answer")
            return ("Aucun resultat", None, RLMStatus.FINAL_ANSWER, "Mot magique non trouve")
    
    # ── Strategie Aggregation (type OOLONG) ──
    def _strat_aggregation(self, query, context, it):
        if it == 0:
            return (
                "Probing: comprendre la structure des donnees",
                '''lines = context.strip().split("\n")
data_lines = [l for l in lines if l.strip() and not l.startswith("#")]
print(f"Total lignes: {len(lines)}, lignes de donnees: {len(data_lines)}")
for l in data_lines[:5]:
    print(f"  {l[:100]}")''',
                RLMStatus.RUNNING, None
            )
        elif it == 1:
            return (
                "Chunking + sous-appels pour classification semantique (Figure 4b)",
                '''lines = context.strip().split("\n")
data_lines = [l for l in lines if l.strip() and not l.startswith("#")]
batch_size = max(1, len(data_lines) // 3)
all_classifications = []
for i in range(0, len(data_lines), batch_size):
    batch = data_lines[i:i+batch_size]
    batch_text = "\n".join(batch)
    # SOUS-APPEL RECURSIF par batch
    result = llm_query(
        f"Classifie chaque ligne par categorie "
        f"(numeric_value, human_being, location, description):\n{batch_text}"
    )
    all_classifications.append(result)
    print(f"Batch {i//batch_size+1}: {len(batch)} lignes traitees")
print(f"Total: {len(all_classifications)} batches classifies")''',
                RLMStatus.RUNNING, None
            )
        elif it == 2:
            return (
                "Agregation PROGRAMMATIQUE — code Python, pas de LLM !",
                '''category_counts = Counter()
for classif in all_classifications:
    for cat in ["numeric_value", "human_being", "location", "description"]:
        category_counts[cat] += classif.lower().count(cat)
final_result = "Resultats d\'agregation:\n"
for cat, count in category_counts.most_common():
    final_result += f"  {cat}: {count} occurrences\n"
print(final_result)''',
                RLMStatus.RUNNING, None
            )
        else:
            return ("Agregation terminee", None, RLMStatus.FINAL_VAR, "final_result")
    
    # ── Strategie generique ──
    def _strat_generic(self, query, context, it):
        if it == 0:
            return (
                "Probing generique",
                'print(f"Contexte: {len(context)} chars")\nprint(context[:300])',
                RLMStatus.RUNNING, None
            )
        elif it == 1:
            return (
                "Chunking + sous-appels",
                f'''chunk_size = min(len(context), 2000)
n = max(1, len(context) // chunk_size)
answers = []
for i in range(n):
    chunk = context[i*chunk_size:(i+1)*chunk_size]
    ans = llm_query(f"Resume: {{chunk}}")
    answers.append(ans)
    print(f"Chunk {{i+1}}/{{n}}: {{ans[:80]}}")
final_answer = llm_query(
    f"Agrege pour repondre a: {query}\n" + "\n".join(answers)
)
print(f"Reponse: {{final_answer}}")''',
                RLMStatus.RUNNING, None
            )
        else:
            return ("Termine", None, RLMStatus.FINAL_VAR, "final_answer")
    
    # ── Affichage ──
    def _show_step(self, it, thought, code_str, execution):
        print(f"\n{'---'*15}")
        print(f"  Iteration {it} | {thought}")
        print(f"{'---'*15}")
        print("  Code REPL:")
        for line in code_str.strip().split("\n"):
            print(f"    | {line}")
        if execution.stdout:
            print("  Sortie:")
            for line in execution.stdout.strip().split("\n")[:12]:
                print(f"    | {line}")
        if execution.stderr:
            print(f"  ERREUR: {execution.stderr[:150]}")
    
    def _stats(self, repl):
        return {
            "root_llm": self.root_llm.stats(),
            "sub_llm": self.sub_llm.stats(),
            "sous_appels_recursifs": repl.sub_call_count,
            "executions_repl": len(repl.execution_history),
        }

print("RecursiveLanguageModel defini")


RecursiveLanguageModel defini


---
## 4. Générateurs de données de test

Deux types de benchmarks simulés, correspondant aux tâches du papier :

- **Haystack** (type S-NIAH) — un long texte avec une "aiguille" cachée → complexité **constante**
- **Données d'agrégation** (type OOLONG) — des entrées à classifier et compter → complexité **linéaire**


In [14]:
def generate_haystack(size: int, needle_position: float = 0.7) -> str:
    """Genere un texte long avec un 'needle' cache (benchmark S-NIAH)."""
    fillers = [
        "La photosynthese convertit la lumiere en energie chimique dans les plantes.",
        "Les algorithmes de tri se classent en O(n log n) ou O(n^2) en complexite.",
        "L'architecture des microprocesseurs utilise le pipeline d'instructions.",
        "La theorie des graphes est fondamentale en informatique.",
        "Les reseaux convolutifs excellent en traitement d'images.",
        "Le protocole TCP/IP est la base d'Internet.",
        "Les bases relationnelles utilisent SQL pour les requetes.",
        "La mecanique quantique decrit les particules subatomiques.",
        "Le reinforcement learning permet l'apprentissage par essai-erreur.",
        "Les systemes distribues gerent coherence et disponibilite.",
    ]
    lines, current = [], 0
    needle_pos = int(size * needle_position)
    needle_inserted = False
    while current < size:
        if not needle_inserted and current >= needle_pos:
            lines.append(">>> Le mot magique secret est: ZEPHYR <<<")
            needle_inserted = True
        else:
            lines.append(fillers[len(lines) % len(fillers)])
        current += len(lines[-1]) + 1
    if not needle_inserted:
        lines.insert(len(lines)//2, ">>> Le mot magique secret est: ZEPHYR <<<")
    return "\n".join(lines)


def generate_aggregation_data(n_entries: int) -> str:
    """Genere des donnees type OOLONG (classification semantique)."""
    questions = [
        "Combien d'etoiles y a-t-il dans la Voie Lactee ?",
        "Qui est le president de la France en 2025 ?",
        "Ou se trouve le Mont Everest ?",
        "Qu'est-ce que la relativite generale ?",
        "Quel nombre represente Pi arrondi a 2 decimales ?",
        "Quelle personne a invente le telephone ?",
        "Quel pays a la plus grande superficie ?",
        "Qu'est-ce que le machine learning ?",
        "Combien de chromosomes a un etre humain ?",
        "Qui est Marie Curie ?",
        "Ou est situee la Grande Muraille ?",
        "Qu'est-ce que la democratie ?",
    ]
    lines = ["# Donnees - classifiez chaque question par categorie semantique"]
    for i in range(n_entries):
        uid = 10000 + (i * 7) % 50000
        q = questions[i % len(questions)]
        lines.append(f"Date: 2024-{(i%12)+1:02d}-{(i%28)+1:02d} || User: {uid} || Instance: {q}")
    return "\n".join(lines)

# Apercu
haystack = generate_haystack(10_000)
agg_data = generate_aggregation_data(30)
print(f"Haystack: {len(haystack):,} chars, {len(haystack.splitlines())} lignes")
print(f"Agregation: {len(agg_data):,} chars, {len(agg_data.splitlines())} lignes")


Haystack: 10,029 chars, 161 lignes
Agregation: 2,585 chars, 31 lignes


---
## 5. Démo 1 — LLM classique vs RLM : Needle in a Haystack

On crée un contexte **2.5× plus grand** que la fenêtre du LLM.

- Le **LLM direct** échoue car le prompt ne rentre pas dans la fenêtre.
- Le **RLM** réussit car il n'injecte **jamais** le prompt complet — il l'explore par code.

C'est exactement ce que montre la **Figure 1** du papier : GPT-5 échoue au-delà de 272K tokens, mais RLM(GPT-5) continue à fonctionner au-delà de 1M tokens.


In [15]:
# ===  LLM DIRECT -> echoue  ===

context_size = 10_000   # 10K caracteres
llm_window   = 4_000    # LLM ne gere que 4K

haystack = generate_haystack(context_size, needle_position=0.7)
query = "Quel est le mot magique secret cache dans le texte ?"

print(f"Contexte: {len(haystack):,} chars")
print(f"Fenetre LLM: {llm_window:,} chars")
print(f"Ratio: {len(haystack)/llm_window:.1f}x la fenetre\n")

direct_llm = SimulatedLLM("gpt-direct", context_window=llm_window)
result = direct_llm(f"{query}\n\n{haystack}")

print(f"LLM direct: {result}")
print("-> Le prompt complet NE RENTRE PAS dans la fenetre de contexte !")


Contexte: 10,029 chars
Fenetre LLM: 4,000 chars
Ratio: 2.5x la fenetre

LLM direct: [ERREUR: Prompt depasse la fenetre de contexte]
-> Le prompt complet NE RENTRE PAS dans la fenetre de contexte !


In [16]:
# ===  RLM -> reussit  ===

root_llm = SimulatedLLM("gpt-root", context_window=llm_window)
sub_llm  = SimulatedLLM("gpt-sub",  context_window=llm_window)

rlm = RecursiveLanguageModel(
    root_llm=root_llm,
    sub_llm=sub_llm,
    max_iterations=6,
    verbose=True,
)

result = rlm(prompt=haystack, query=query)



  RLM | 10,029 chars | Query: Quel est le mot magique secret cache dans le texte ?...
  Root: gpt-root (fenetre 4,000)
  Sub:  gpt-sub

---------------------------------------------
  Iteration 0 | Probing: examiner la structure du contexte
---------------------------------------------
  Code REPL:
    | print(f"Taille: {len(context)} chars, {len(context.splitlines())} lignes")
    | print(f"Debut: {context[:150]}...")
    | print(f"Fin: ...{context[-150:]}")
  Sortie:
    | Taille: 10029 chars, 161 lignes
    | Debut: La photosynthese convertit la lumiere en energie chimique dans les plantes.
    | Les algorithmes de tri se classent en O(n log n) ou O(n^2) en complexite.
    | ...
    | Fin: ...r essai-erreur.
    | Les systemes distribues gerent coherence et disponibilite.
    | La photosynthese convertit la lumiere en energie chimique dans les plantes.

---------------------------------------------
  Iteration 1 | Filtrage par priors: recherche regex ciblee (pattern Figure 4a)
----

In [17]:
# Analyse de la trajectoire
print("Statistiques de l'execution RLM:")
print(f"   Iterations:                  {len(result.trajectory)}")
print(f"   Sous-appels LLM recursifs:   {result.stats['sous_appels_recursifs']}")
print(f"   Executions REPL:             {result.stats['executions_repl']}")
print(f"   Chars traites (sous-LLM):    {result.stats['sub_llm']['chars_traites']:,}")
print(f"   Chars du contexte original:  {len(haystack):,}")
print(f"   Ratio d'efficacite:          {result.stats['sub_llm']['chars_traites']/len(haystack):.1%}")
print(f"\nReponse: {result.answer}")


Statistiques de l'execution RLM:
   Iterations:                  4
   Sous-appels LLM recursifs:   0
   Executions REPL:             3
   Chars traites (sous-LLM):    0
   Chars du contexte original:  10,029
   Ratio d'efficacite:          0.0%

Reponse: Mot magique non trouve


### 💡 Observation clé

Le RLM n'a traité qu'une **fraction** du contexte total via le sous-LLM. Le reste a été filtré par du **code** (regex) sans jamais passer par un LLM.

C'est exactement l'**Observation 1** du papier :
> *"RLMs scale well to the theoretical costs of extending a base model's context window — on BrowseComp-Plus (1K), the cost of GPT-5-mini ingesting 6-11M input tokens is \$1.50 – \$2.75, while RLM(GPT-5) has an average cost of \$0.99."*


---
## 6. Démo 2 — Tâche d'agrégation (type OOLONG)

Le benchmark **OOLONG** exige de traiter **chaque ligne** du contexte (complexité linéaire).

La stratégie du RLM :
1. **Probing** → comprendre le format des données
2. **Chunking + sous-appels** → classifier chaque batch via un sous-LLM
3. **Agrégation par code** → compter avec Python (pas le LLM !)

C'est le pattern de la **Figure 4b-c** du papier.


In [18]:
agg_data = generate_aggregation_data(30)

root_llm2 = SimulatedLLM("gpt-root", context_window=8_000)
sub_llm2  = SimulatedLLM("gpt-sub",  context_window=4_000)

rlm2 = RecursiveLanguageModel(
    root_llm=root_llm2,
    sub_llm=sub_llm2,
    max_iterations=6,
    verbose=True,
)

result2 = rlm2(prompt=agg_data, query="Classifie chaque question et compte combien par categorie.")



  RLM | 2,585 chars | Query: Classifie chaque question et compte combien par categorie....
  Root: gpt-root (fenetre 8,000)
  Sub:  gpt-sub

---------------------------------------------
  Iteration 0 | Probing: comprendre la structure des donnees
---------------------------------------------
  Code REPL:
    | lines = context.strip().split("
    | ")
    | data_lines = [l for l in lines if l.strip() and not l.startswith("#")]
    | print(f"Total lignes: {len(lines)}, lignes de donnees: {len(data_lines)}")
    | for l in data_lines[:5]:
    |     print(f"  {l[:100]}")
  ERREUR: Traceback (most recent call last):
  File "/tmp/ipykernel_223055/99369750.py", line 53, in execute
    exec(code, self._namespace)
  File "<string>", 

---------------------------------------------
  Iteration 1 | Chunking + sous-appels pour classification semantique (Figure 4b)
---------------------------------------------
  Code REPL:
    | lines = context.strip().split("
    | ")
    | data_lines = [l for l 

In [19]:
print("Statistiques agregation:")
print(f"   Sous-appels recursifs: {result2.stats['sous_appels_recursifs']}")
print(f"   (= nombre de batches envoyes au sous-LLM)")
print(f"\nResultat:\n{result2.answer}")


Statistiques agregation:
   Sous-appels recursifs: 0
   (= nombre de batches envoyes au sous-LLM)

Resultat:
[Variable introuvable]


### 💡 Séparation code / LLM

Notez comment le RLM utilise **chaque outil pour ce qu'il fait le mieux** :
- Le **sous-LLM** pour la classification sémantique (requiert du "raisonnement")
- Le **code Python** pour l'agrégation (compter, trier = trivial en code, fragile en LLM)


---
## 7. Visualisation d'une trajectoire RLM

Chaque exécution produit une **trajectoire** — la séquence de pensées, code, et résultats. C'est analogue à un "plan d'exécution" en base de données.


In [20]:
def show_trajectory(rlm_result: RLMResult, title: str = "Trajectoire"):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")
    
    for step in rlm_result.trajectory:
        icon = {
            RLMStatus.RUNNING: "[...]",
            RLMStatus.FINAL_ANSWER: "[OK]",
            RLMStatus.FINAL_VAR: "[OK]",
        }[step.status]
        
        print(f"\n  {icon} Step {step.iteration}: {step.thought}")
        
        if step.code:
            sub_calls = step.code.count("llm_query(")
            code_lines = len(step.code.strip().split("\n"))
            print(f"       {code_lines} lignes de code, {sub_calls} sous-appel(s) LLM")
        
        if step.execution and step.execution.stdout:
            first_line = step.execution.stdout.strip().split("\n")[0]
            print(f"       -> {first_line}")
    
    print(f"\n  Reponse: {rlm_result.answer[:100]}")
    print(f"  Cout: {rlm_result.stats['sous_appels_recursifs']} sous-appels, "
          f"{rlm_result.stats['executions_repl']} executions REPL")
    print(f"{'='*55}")

show_trajectory(result, "Trajectoire NIAH (Needle in a Haystack)")
show_trajectory(result2, "Trajectoire Agregation (type OOLONG)")



  Trajectoire NIAH (Needle in a Haystack)

  [...] Step 0: Probing: examiner la structure du contexte
       3 lignes de code, 0 sous-appel(s) LLM
       -> Taille: 10029 chars, 161 lignes

  [...] Step 1: Filtrage par priors: recherche regex ciblee (pattern Figure 4a)
       6 lignes de code, 0 sous-appel(s) LLM
       -> 'magique' trouve a pos 7025: ...lgorithmes de tri se classent en O(n log n) ou O(n^2) en complexite.

  [...] Step 2: Sous-appel recursif sur le fragment trouve (coeur du RLM)
       20 lignes de code, 2 sous-appel(s) LLM

  [OK] Step 3: Aucun resultat

  Reponse: Mot magique non trouve
  Cout: 0 sous-appels, 3 executions REPL

  Trajectoire Agregation (type OOLONG)

  [...] Step 0: Probing: comprendre la structure des donnees
       6 lignes de code, 0 sous-appel(s) LLM

  [...] Step 1: Chunking + sous-appels pour classification semantique (Figure 4b)
       18 lignes de code, 1 sous-appel(s) LLM

  [...] Step 2: Agregation PROGRAMMATIQUE — code Python, pas de LLM 

---
## 8. Résumé : pourquoi le RLM fonctionne

### Le problème fondamental

| | LLM classique | RLM |
|---|---|---|
| **Prompt** | Injecté dans le Transformer | Stocké comme variable REPL |
| **Fenêtre** | Limite dure | Aucune limite théorique |
| **Context rot** | Inévitable sur longs contextes | Évité par accès sélectif |
| **Coût** | Proportionnel à la taille du prompt | Proportionnel à la complexité de la tâche |

### Les 3 ingrédients clés

1. **Environnement REPL** — Le prompt est un objet manipulable, pas une entrée neuronale
2. **Sous-appels récursifs** — Le LLM peut s'invoquer lui-même sur des fragments ciblés  
3. **Hybridation code/LLM** — Code pour le structurel, LLM pour le sémantique

### Résultats du papier (Table 1)

| Tâche | GPT-5 seul | RLM(GPT-5) | Gain |
|---|---|---|---|
| OOLONG | 44.00 | **56.50** | +28.4% |
| OOLONG-Pairs | 0.04 | **58.00** | +57.96 pts |
| BrowseComp+ (1K) | 0.00* | **91.33** | de 0 à 91% |
| CodeQA | 24.00* | **62.00** | +38 pts |

*\* = le prompt ne rentre pas dans la fenêtre de contexte*

### Pour aller plus loin

- Remplacer `SimulatedLLM` par de vrais appels API (OpenAI, Anthropic)
- Implémenter la profondeur de récursion > 1 (sous-RLMs au lieu de sous-LLMs)
- Ajouter de l'asynchronisme dans les sous-appels pour réduire la latence
- Entraîner un modèle spécifiquement pour le mode RLM (piste mentionnée §5 du papier)
